# Modul 6: Model Eksogen Cuaca (SARIMAX) & Multivariat VAR
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 6 (Minggu 6):** Mengintegrasikan variabel eksogen cuaca (temperatur & kelembaban) pada SARIMAX menggunakan dataset terkurasi `beban_cuaca_regional_168h.csv` serta memodelkan interaksi multivariat dengan VAR & *Impulse Response Function* (IRF).

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

warnings.filterwarnings('ignore')
plt.rcParams['axes.grid'] = True

## 1. Memuat Dataset Beban & Cuaca Regional (`beban_cuaca_regional_168h.csv`)
Notebook ini otomatis mencari dataset secara lokal (`datasets/`), lalu mengunduh dari repositori GitHub resmi saat dijalankan di Google Colab, dengan fallback generator sintetis jika luring.

In [ ]:
CSV_NAME = 'beban_cuaca_regional_168h.csv'
RAW_URL = 'https://raw.githubusercontent.com/novatha/deret-waktu-tee412/main/labs/datasets/' + CSV_NAME

def load_load_weather():
    for p in [Path('datasets') / CSV_NAME, Path('labs/datasets') / CSV_NAME]:
        if p.exists():
            print(f"Memuat dataset lokal: {p}")
            return pd.read_csv(p, parse_dates=['Timestamp'], index_col='Timestamp')
    try:
        print(f"Mengunduh dataset dari GitHub: {RAW_URL}")
        return pd.read_csv(RAW_URL, parse_dates=['Timestamp'], index_col='Timestamp')
    except Exception as e:
        print(f"Fallback sintetis (alasan: {e})")
        rng_fb = np.random.default_rng(66)
        n = 168
        idx = pd.date_range('2026-09-07', periods=n, freq='h')
        hr = idx.hour
        temp = 28 + 5.5 * np.sin(2 * np.pi * (hr - 8) / 24) + rng_fb.normal(0, 1.0, n)
        hum = 75 - 12 * np.sin(2 * np.pi * (hr - 8) / 24) + rng_fb.normal(0, 2.5, n)
        load = 220 + 4.2 * temp + 0.25 * hum + 45 * np.sin(2 * np.pi * (hr - 7) / 24) + rng_fb.normal(0, 4.5, n)
        return pd.DataFrame({'Load_MW': load, 'Temp_C': temp, 'Humidity_Pct': hum, 'Is_Weekend': (idx.dayofweek >= 5).astype(int)}, index=idx)

df = load_load_weather().asfreq('h')
print(df.head(3))
train, test = df.iloc[:-24], df.iloc[-24:]  # 6 hari latih (144 jam), 1 hari uji (24 jam)

## 2. Komparasi SARIMA (Tanpa Cuaca) vs. SARIMAX (Dengan Eksogen Cuaca)

In [ ]:
m_base = SARIMAX(train['Load_MW'], order=(1, 0, 1), seasonal_order=(1, 1, 0, 24)).fit(disp=False)
m_exog = SARIMAX(
    train['Load_MW'], exog=train[['Temp_C', 'Humidity_Pct']],
    order=(1, 0, 1), seasonal_order=(1, 1, 0, 24)
).fit(disp=False)

fc_base = m_base.forecast(steps=24)
fc_exog = m_exog.forecast(steps=24, exog=test[['Temp_C', 'Humidity_Pct']])

rmse_base = np.sqrt(np.mean((test['Load_MW'] - fc_base) ** 2))
rmse_exog = np.sqrt(np.mean((test['Load_MW'] - fc_exog) ** 2))
print(f"Koefisien Eksogen SARIMAX -> Temp_C: {m_exog.params['Temp_C']:.3f} MW/°C | Humidity_Pct: {m_exog.params['Humidity_Pct']:.3f}")
print(f"RMSE SARIMA (tanpa cuaca) : {rmse_base:.2f} MW (AIC={m_base.aic:.1f})")
print(f"RMSE SARIMAX (dg cuaca)   : {rmse_exog:.2f} MW (AIC={m_exog.aic:.1f})")

plt.figure(figsize=(11, 4))
plt.plot(test.index, test['Load_MW'], 'k-o', ms=4, label='Aktual Beban (MW)')
plt.plot(test.index, fc_base, '--', color='#f59e0b', lw=1.8, label=f'SARIMA (RMSE={rmse_base:.2f})')
plt.plot(test.index, fc_exog, '-', color='#10b981', lw=2.0, label=f'SARIMAX + Cuaca (RMSE={rmse_exog:.2f})')
plt.title('Peramalan Beban 24 Jam: SARIMA vs. SARIMAX (Dataset Regional)')
plt.ylabel('Beban (MW)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Pemodelan Multivariat VAR & Impulse Response Function (IRF)

In [ ]:
# Simulasi sistem 2-kanal: Arus Beban (A) mempengaruhi Jatuh Tegangan Bus (V)
rng = np.random.default_rng(66)
n_var = 250
arus = np.zeros(n_var)
tegangan = np.zeros(n_var)
for t in range(1, n_var):
    arus[t] = 0.65 * arus[t - 1] + rng.normal(0, 2.0)
    tegangan[t] = 0.50 * tegangan[t - 1] - 0.35 * arus[t - 1] + rng.normal(0, 0.5)

df_pq = pd.DataFrame({'arus': arus, 'tegangan': tegangan})
res_var = VAR(df_pq).fit(maxlags=8, ic='aic')
print(f"Ordo VAR terpilih (AIC): p = {res_var.k_ar}")

irf = res_var.irf(10)
irf.plot(impulse='arus', response='tegangan')
plt.suptitle('Impulse Response Function: Dampak Lonjakan Arus terhadap Tegangan Bus', y=1.02)
plt.show()

## 4. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Mengapa koefisien `Temp_C` berkorelasi positif terhadap beban listrik dan bagaimana pengaruhnya terhadap RMSE/AIC SARIMAX?
2. **[C5]** Apa risiko praktis SARIMAX di lapangan jika ramalan temperatur BMKG 24 jam ke depan meleset $\pm 3^\circ\text{C}$? Uji dengan menambahkan derau pada `test['Temp_C']`!